# 02 — ETL dataset Reddit Data Science Posts

Nguồn đã audit: `reddit_database.csv`, 545,427 dòng, 13 cột, 340,112,365 bytes; thời gian 2008-03-19 đến 2022-05-08 UTC. Khoảng 49.7% `post` bị thiếu nhưng `title` không thiếu; do đó tạo văn bản phân tích bằng cách ghép title và post, không loại bài chỉ vì post rỗng. Bỏ `author` khỏi bảng phân tích vì không cần định danh người dùng cho các câu hỏi nghiên cứu hiện tại.

Notebook đọc theo chunks để giới hạn RAM, giữ raw nguyên bản và tạo Parquet sạch cùng thống kê chất lượng.


In [ ]:
%pip -q install pandas pyarrow


In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATASET_HANDLE = "maksymshkliarevskyi/reddit-data-science-posts"
RAW_ROOT = Path("/content/data/raw/reddit-data-science-posts")
OUT_DIR = Path("/content/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PARQUET = OUT_DIR / "reddit_posts_clean.parquet"
REPORT_PATH = OUT_DIR / "etl_report.json"

# Colab có thể trả file Kaggle vào cache khác với thư mục mình dự kiến.
# Tìm file trước trong /content; nếu runtime mới chưa có file thì tải lại.
candidates = list(Path("/content").rglob("reddit_database.csv"))
if not candidates:
    import kagglehub
    RAW_ROOT.mkdir(parents=True, exist_ok=True)
    kagglehub.dataset_download(DATASET_HANDLE, output_dir=str(RAW_ROOT))
    candidates = list(Path("/content").rglob("reddit_database.csv"))
if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy reddit_database.csv trong /content sau khi tải. "
        "Hãy chạy lại notebook 01_data_audit.ipynb và kiểm tra bước tải Kaggle."
    )
RAW_CSV = candidates[0]
print("Đang dùng file raw:", RAW_CSV)
if len(candidates) > 1:
    print("Có nhiều bản CSV; đang dùng bản đầu tiên:", candidates)


In [ ]:
def parse_reddit_time(series):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().mean() > 0.8:
        median_abs = numeric.dropna().abs().median()
        unit = "ms" if median_abs > 1e12 else "s" if median_abs > 1e9 else None
        return pd.to_datetime(numeric, errors="coerce", utc=True, unit=unit)
    return pd.to_datetime(series, errors="coerce", utc=True)

def make_text(title, post):
    title = title.fillna("").astype(str).str.strip()
    post = post.fillna("").astype(str).str.strip()
    text = title.where(title.eq(""), "Title: " + title)
    both = title.ne("") & post.ne("")
    only_post = title.eq("") & post.ne("")
    text.loc[both] = text.loc[both] + "\n\nPost: " + post.loc[both]
    text.loc[only_post] = "Post: " + post.loc[only_post]
    return text.str.strip()


In [ ]:
# ETL theo chunks; de-duplicate ID xuyên suốt các chunks.
chunksize = 50_000
seen_ids = set()
writer = None
counts = {"input_rows": 0, "output_rows": 0, "duplicate_ids_removed": 0, "empty_text_removed": 0, "invalid_timestamp_removed": 0}
missing_counts = {}
subreddit_counts = {}
monthly_counts = {}

usecols = ["created_date", "created_timestamp", "subreddit", "title", "id", "score", "num_comments", "num_crossposts", "subreddit_subscribers", "post"]

try:
    for chunk in pd.read_csv(RAW_CSV, usecols=lambda c: c in usecols, chunksize=chunksize, low_memory=False):
        counts["input_rows"] += len(chunk)
        for col, n in chunk.isna().sum().items():
            missing_counts[col] = missing_counts.get(col, 0) + int(n)

        # Prefer created_timestamp; use created_date only where it fails to parse.
        timestamp = parse_reddit_time(chunk["created_timestamp"])
        bad_time = timestamp.isna()
        if bad_time.any():
            fallback = parse_reddit_time(chunk.loc[bad_time, "created_date"])
            timestamp.loc[bad_time] = fallback

        text = make_text(chunk["title"], chunk["post"])
        empty_text = text.eq("")
        invalid_time = timestamp.isna()

        id_text = chunk["id"].fillna("").astype(str).str.strip()
        duplicate = id_text.isin(seen_ids) | id_text.eq("")
        counts["duplicate_ids_removed"] += int(duplicate.sum())
        seen_ids.update(id_text.loc[~duplicate].tolist())
        counts["empty_text_removed"] += int((empty_text & ~duplicate).sum())
        counts["invalid_timestamp_removed"] += int((invalid_time & ~duplicate & ~empty_text).sum())

        keep = ~duplicate & ~empty_text & ~invalid_time
        clean = pd.DataFrame({
            "id": id_text.loc[keep].astype(str).values,
            "subreddit": chunk.loc[keep, "subreddit"].astype(str).str.strip().values,
            "created_at_utc": timestamp.loc[keep].values,
            "text": text.loc[keep].values,
            "score": pd.to_numeric(chunk.loc[keep, "score"], errors="coerce").astype("float64").values,
            "num_comments": pd.to_numeric(chunk.loc[keep, "num_comments"], errors="coerce").astype("Int64").values,
            "num_crossposts": pd.to_numeric(chunk.loc[keep, "num_crossposts"], errors="coerce").astype("Int64").values,
            "subreddit_subscribers": pd.to_numeric(chunk.loc[keep, "subreddit_subscribers"], errors="coerce").astype("Int64").values,
        })
        counts["output_rows"] += len(clean)
        if len(clean):
            clean["created_at_utc"] = pd.to_datetime(clean["created_at_utc"], utc=True)
            clean["subreddit"] = clean["subreddit"].astype("string")
            clean["id"] = clean["id"].astype("string")
            clean["text"] = clean["text"].astype("string")
            table = pa.Table.from_pandas(clean, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(OUT_PARQUET, table.schema, compression="zstd")
            writer.write_table(table)
            for name, n in clean["subreddit"].value_counts().items():
                subreddit_counts[str(name)] = subreddit_counts.get(str(name), 0) + int(n)
            by_month = clean["created_at_utc"].dt.strftime("%Y-%m").value_counts()
            for month, n in by_month.items():
                monthly_counts[month] = monthly_counts.get(month, 0) + int(n)
finally:
    if writer is not None:
        writer.close()

if writer is None:
    raise ValueError("Không còn bản ghi hợp lệ sau ETL. Kiểm tra định dạng timestamp và văn bản.")

print(json.dumps(counts, ensure_ascii=False, indent=2))
print("Số subreddit hợp lệ:", len(subreddit_counts))
print("Parquet:", OUT_PARQUET, f"({OUT_PARQUET.stat().st_size:,} bytes)")


In [ ]:
report = {
    "source_file": str(RAW_CSV),
    "output_file": str(OUT_PARQUET),
    "counts": counts,
    "missing_in_source": missing_counts,
    "subreddit_counts": dict(sorted(subreddit_counts.items(), key=lambda kv: kv[1], reverse=True)),
    "monthly_counts": dict(sorted(monthly_counts.items())),
    "clean_schema": str(pq.read_schema(OUT_PARQUET)),
}
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("ETL report:", REPORT_PATH)
print("Số tháng có dữ liệu:", len(monthly_counts))
print("5 subreddit nhiều bài nhất:")
print(pd.Series(subreddit_counts).sort_values(ascending=False).head())


In [ ]:
# Đọc lại một mẫu để kiểm tra schema và nội dung đầu ra
preview = pd.read_parquet(OUT_PARQUET)
print(preview.shape)
display(preview.head(3))
print(preview.dtypes)


## Lưu ý sau khi chạy

- Đối chiếu `input_rows = output_rows + duplicate_ids_removed + empty_text_removed + invalid_timestamp_removed` (các nhóm loại được tính theo thứ tự ID trùng, văn bản rỗng, timestamp lỗi).
- Kiểm tra `monthly_counts`: khoảng thời gian kéo dài nhiều năm có thể có độ phủ không đều. Trước khi kết luận xu hướng, vẽ số bài theo tháng/năm và cân nhắc chuẩn hóa theo subreddit.
- `num_crossposts` và `subreddit_subscribers` giữ null khi thiếu; không tự đổi null thành 0.
- Tải `etl_report.json` và Parquet về Drive nếu muốn giữ output sau khi Colab ngắt phiên.
